In [1]:
import os
import sys

import numpy as np
import scipy

In [2]:
sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
import topnum

from topnum.scores.perplexity_score import PerplexityScore
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.model_constructor import init_model_from_family, KnownModel, PARAMS_EXPLORED, init_plsa

In [4]:
import artm
from artm import ARTM, Dictionary

import topicnet
from topicnet.cooking_machine.dataset import Dataset
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)

from topicnet.cooking_machine.models.topic_model import ARTM_NINE
from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.viewers.top_documents_viewer import TopDocumentsViewer
from topicnet.viewers.top_tokens_viewer import TopTokensViewer
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    count_vocab_size,
    init_model,
)
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    modality_weight_rel2abs,
    transform_regularizer,
)


import numpy as np
import pandas as pd
from pandas import DataFrame
from scipy.spatial.distance import cdist

import os
import tempfile
import warnings
from copy import deepcopy
from typing import Dict, List, Optional

In [5]:
from IPython.display import display, display_html

In [6]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [7]:
! ls $DATA_FOLDER_PATH

20NG.csv	 MKB10__internals      RTL_Wiki_person.csv
20NG__internals  postnauka.csv	       RTL_Wiki_person__internals
api.py		 postnauka__internals  ruwiki_good__internals
Brown		 __pycache__	       ruwiki_good.txt
Brown_BOW.csv	 Reuters	       WikiRef-220
Brown_NOOW.csv	 Reuters_BOW.csv       wiki_ref220_bow.csv
__init__.py	 Reuters_NOOW.csv      wiki_ref220_natural_order.csv
MKB10.csv	 RTL_Wiki.csv


In [8]:
dataset = Dataset(
    f'{DATA_FOLDER_PATH}/20NG.csv',
)

dataset.get_possible_modalities()

{'@bigram', '@lemmatized'}

In [9]:
MAIN_MODALITY = '@lemmatized'

In [10]:
dataset._data.head()

,Unnamed: 0,raw_text,filenames,target,id,tokenized,lemmatized,bigram,vw_text
id,,,,,,,,,
rec_autos_102994,0,I was wondering if anyone out there could enli...,/home/egorov/scikit_learn_data/20news_home/20n...,7,rec_autos_102994,"[('was', 'VBD'), ('wondering', 'VBG'), ('if', ...","['wonder', 'anyone', 'could', 'enlighten', 'ca...","['wonder_anyone', 'anyone_could', 'sport_car',...",rec_autos_102994 |@lemmatized wonder:1 anyone:...
comp_sys_mac_hardware_51861,1,A fair number of brave souls who upgraded thei...,/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51861,"[('fair', 'JJ'), ('number', 'NN'), ('of', 'IN'...","['fair', 'number', 'brave', 'soul', 'upgrade',...","['clock_oscillator', 'please_send', 'top_speed...",comp_sys_mac_hardware_51861 |@lemmatized fair:...
comp_sys_mac_hardware_51879,2,"well folks, my mac plus finally gave up the gh...",/home/egorov/scikit_learn_data/20news_home/20n...,4,comp_sys_mac_hardware_51879,"[('well', 'RB'), ('folks', 'NNS'), ('my', 'PRP...","['well', 'folk', 'mac', 'plus', 'finally', 'gi...","['mac_plus', 'life_way', 'way_back', 'market_n...",comp_sys_mac_hardware_51879 |@lemmatized well:...
comp_graphics_38242,3,\nDo you have Weitek's address/phone number? ...,/home/egorov/scikit_learn_data/20news_home/20n...,1,comp_graphics_38242,"[('do', 'VBP'), ('you', 'PRP'), ('have', 'VB')...","['weitek', 'address', 'phone', 'number', 'like...","['address_phone', 'phone_number', 'number_like...",comp_graphics_38242 |@lemmatized weitek:1 addr...
sci_space_60880,4,"From article <C5owCB.n3p@world.std.com>, by to...",/home/egorov/scikit_learn_data/20news_home/20n...,14,sci_space_60880,"[('from', 'IN'), ('article', 'NN'), ('by', 'IN...","['article', 'tom', 'baker', 'understanding', '...","['system_software', 'thing_check', 'introduce_...",sci_space_60880 |@lemmatized article:1 tom:1 b...


In [29]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        _T, W = phi.shape
        T = len(topic_indices)

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [12]:
%%time

occurences, co_occurences = calc_doc_occurrences(dataset, MAIN_MODALITY)

CPU times: user 2.63 s, sys: 127 ms, total: 2.76 s
Wall time: 2.72 s


In [15]:
co_occurences

<52744x52744 sparse matrix of type '<class 'numpy.int64'>'
	with 72331838 stored elements in Compressed Sparse Row format>

In [16]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    dataset.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [17]:
class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

In [18]:
def view_model(
        topic_model,
        dataset,
        num_top_tokens: int = 5,
        top_tokens_method: str = 'phi',
        num_topics: Optional[int] = 5,  # we do not want to fill the whole .ipynb notebook with topics...
        ):
    top_tok_viewer = TopTokensViewer(
        topic_model, num_top_tokens=num_top_tokens, method=top_tokens_method
    )
    top_doc_viewer = TopDocumentsViewer(topic_model, dataset=dataset)
    top_docs = top_doc_viewer.view()

    if num_topics is None:
        num_topics = len(topic_model.topic_names)

    for topic_name in topic_model.topic_names[:num_topics]:
        topic_top_toks = top_tok_viewer.to_html(topic_names=[topic_name])
        topic_top_docs = top_docs[topic_name]
        display_html(topic_top_toks, raw=True)
        display(topic_top_docs)

In [19]:
class FastFixPhiRegularizer(BaseRegularizer):
    _VERY_BIG_TAU = 10 ** 9

    def __init__(self, name: str, parent_model, topic_names: List[str]):
        super().__init__(name, tau=self._VERY_BIG_TAU)

        self._topic_names = topic_names
        self._topic_indices = None
        self._parent_model = parent_model

    def grad(self, pwt, nwt):
        # print('Fixing')

        rwt = np.zeros_like(pwt)
        parent_phi = self._parent_model.get_phi()
        
        rwt[:, self._topic_indices] += parent_phi.values[:, self._topic_indices]

        return self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = self._model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [20]:
class DecorrelatorWithOtherPhiRegularizer(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._other_topic_sum = self._other_phi.values.sum(
            axis=1, keepdims=True
        )
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        # print('Decorring')

        rwt = np.zeros_like(pwt)
        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * self._other_topic_sum
        )

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [21]:
class DecorrelatorWithOtherPhiRegularizer2(BaseRegularizer):
    def __init__(self, name, tau, topic_names, other_phi, num_iters: Optional[int] = None):
        super().__init__(name, tau=tau)

        self._topic_names = topic_names
        self._other_phi = other_phi
        self._num_iters = num_iters
        self._cur_iter = 0
        
        self._topic_indices = None
        
    def grad(self, pwt, nwt):
        rwt = np.zeros_like(pwt)
        
        if self._num_iters is not None and self._cur_iter >= self._num_iters:
            return rwt

        correlations = cdist(
            self._other_phi.values.T,
            pwt.values[:, self._topic_indices].T,
            lambda u, v: (u * v).sum()
        )
        weighted_other_topics = self._other_phi.values.dot(correlations)

        rwt[:, self._topic_indices] += (
            pwt.values[:, self._topic_indices] * weighted_other_topics
        )
        self._cur_iter += 1

        return -1 * self.tau * rwt

    def attach(self, model):
        super().attach(model)
        
        phi = model.get_phi()
        self._topic_indices = [
            phi.columns.get_loc(topic_name)
            for topic_name in self._topic_names
        ]

In [22]:
NUM_TOPICS = 20  # vary
MAX_NUM_TRAINS = 20
NUM_ITERATIONS = 20
NUM_TOP_TOKENS = 20

In [23]:
def fit_and_compute_scores(model, dataset, custom_regularizers=None):
    print(custom_regularizers)

    model._fit(dataset.get_batch_vectorizer(), num_iterations=NUM_ITERATIONS, custom_regularizers=custom_regularizers)

    score_values = {
        'perplexity': model.scores[f'PerplexityScore{MAIN_MODALITY}'][-1],
    }

    phi = model.get_phi()

    # Currently all topics are taken into account
    target_topic_indices = list(range(NUM_TOPICS))  # phi.columns.get_loc()
    target_topic_names = [phi.columns[i] for i in target_topic_indices]

    top = NUM_TOP_TOKENS
    coherence_score = TopTokenCoherence(
        name=f'coherence_{top}',
        func=create_pmi_top_function(
            occurences, co_occurences,
            dataset.get_dataset().shape[0], [top],
            topic_indices=target_topic_indices,
            co_occurrences_smooth=1e-2,
        )
    )

    value = coherence_score.call(model)
    score_values[coherence_score._name] = value
    topic_coherences = coherence_score.call_by_topic(model)
    topic_coherences = {t: float(v) for t, v in topic_coherences.items()}

    diversity_scores = [
        DiversityScore(
            name=f'diversity_{metric}',
            metric=metric,
            topic_names=target_topic_names,
            class_ids=MAIN_MODALITY,
        )
    
        for metric in KNOWN_METRICS
    ]
    
    for score in diversity_scores:
        value = score.call(model)
        score_values[score._name] = value

    return {
        'scores': score_values,
        'topic_coherences': topic_coherences,
    }

In [24]:
def is_good(coherence):
    # 80 p
    return 1.6095355359760972 <= coherence

def is_bad(coherence):
    # 20 p
    return coherence <= 0.8497888357888863

## Test

In [26]:
model = init_model_from_family(
    family=KnownModel.PLSA,
    dataset=dataset,
    main_modality=MAIN_MODALITY,
    num_topics=NUM_TOPICS,
    seed=2024,
)

In [27]:
result = fit_and_compute_scores(model, dataset)

None


In [28]:
result['topic_coherences']

{0: 1.3237581790853068,
 1: 1.2829826937052067,
 2: 1.490025133820413,
 3: 0.7857971478575373,
 4: 1.3135809496335025,
 5: 3.3417329208687425,
 6: 1.860396670636734,
 7: 0.6210007920538984,
 8: 1.5062729063927354,
 9: 1.5163815087267571,
 10: 0.8543499618407997,
 11: 1.5498711537678034,
 12: 0.9686454389395989,
 13: 0.896240503636927,
 14: 0.5476823963231512,
 15: 1.0634088355376492,
 16: 0.65465596884585,
 17: 1.5921116878827575,
 18: 0.9202599190166817,
 19: 1.1071412340594873}

In [101]:
good_topic_indices = [
    t for t, c in result['topic_coherences'].items() if is_good(c)  # c >= HIGH_COHERENCE_THRESHOLD
]
bad_topic_indices = [
    t for t, c in result['topic_coherences'].items() if is_bad(c)  # c <= LOW_COHERENCE_THRESHOLD
]

phi = model.get_phi()
good_topic_names = [phi.columns[t] for t in good_topic_indices]
bad_topic_names = [phi.columns[t] for t in bad_topic_indices]

In [102]:
len(good_topic_indices), len(bad_topic_indices)

(3, 4)

In [103]:
good_topic_names, bad_topic_names

(['topic_3', 'topic_11', 'topic_15'],
 ['topic_8', 'topic_13', 'topic_18', 'topic_19'])

In [99]:
phi['topic_11'].sort_values(ascending=False)[:20]

modality  token       
@word     россия          0.008695
          война           0.008296
          государство     0.008275
          власть          0.007372
          страна          0.006107
          германия        0.005124
          политический    0.005033
          сталин          0.004779
          стать           0.004387
          революция       0.004353
          политика        0.003816
          франция         0.003641
          партия          0.003525
          военный         0.003510
          сторона         0.003133
          русский         0.003102
          должный         0.003097
          демократия      0.002889
          вопрос          0.002860
          народ           0.002849
Name: topic_11, dtype: float32

In [106]:
fix_regularizer = FastFixPhiRegularizer(
    name='fix',
    parent_model=model._model,
    topic_names=good_topic_names,
)

other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)
decorr_bad_regularizer = DecorrelatorWithOtherPhiRegularizer(
    name='ext_decorr_bad', tau=25,  # 1e5
    topic_names=bad_topic_names,
    other_phi=other_phi
)

other_phi = model._model.get_phi()[good_topic_names]
other_phi = deepcopy(other_phi)
decorr_good_regularizer = DecorrelatorWithOtherPhiRegularizer(
    name='ext_decorr_good', tau=25,  # 1e5
    topic_names=good_topic_names,
    other_phi=other_phi
)

In [107]:
%%time

model._fit(
    dataset.get_batch_vectorizer(),
    num_iterations=10,
    custom_regularizers={
        fix_regularizer.name: fix_regularizer,
        decorr_bad_regularizer.name: decorr_bad_regularizer,
        decorr_good_regularizer.name: decorr_good_regularizer,
    }
)

CPU times: user 19.9 s, sys: 0 ns, total: 19.9 s
Wall time: 10.9 s


In [108]:
other_phi = model._model.get_phi()[bad_topic_names]
other_phi = deepcopy(other_phi)

In [112]:
other_phi.rename(
    columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
)

In [116]:
pd.concat([other_phi, model._model.get_phi(['topic_0'])], axis=1)

,m1_topic_8,m1_topic_13,m1_topic_18,m1_topic_19,topic_0
инвалидность,7.263344e-06,0.000000e+00,0.000000e+00,0.000000e+00,1.535188e-09
мазка,0.000000e+00,0.000000e+00,2.024645e-05,1.530354e-05,9.608340e-14
professor,0.000000e+00,3.533544e-13,1.517497e-12,0.000000e+00,0.000000e+00
умно,1.804371e-11,0.000000e+00,2.047675e-05,1.227452e-15,0.000000e+00
игил,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...
милосердие,1.776537e-05,0.000000e+00,3.757288e-16,2.070272e-14,2.191762e-05
поверка,0.000000e+00,6.251613e-06,2.226501e-05,0.000000e+00,3.302222e-06
вто,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
слоить,1.872259e-05,6.039159e-16,3.578878e-05,0.000000e+00,1.846761e-15


In [23]:
def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [24]:
NUM_TRAINS = 3
TOPIC_INDICES = list(range(NUM_TOPICS))

In [25]:
TOPIC_INDICES

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [25]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer
DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7]


for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

10
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
6 2 14


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d1385040>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d13850d0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d120fa90>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10
ext_decorr_good: 10
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d08e6460>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d0895e50>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d10683d0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.06388419415493585
sparse_theta_sp: -0.29816015719918026
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000.0
ext_decorr_good: 10000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
7 3 13


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d0798760>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d08a3ee0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f96d0dcf730>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.07371253171723367
sparse_theta_sp: -0.3440309506144388
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000.0
ext_decorr_good: 10000000.0


In [30]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 2226.6844889322915
100 2226.6844889322915
1000 2226.6844889322915
10000 2226.6844889322915
100000.0 2226.6844889322915
1000000.0 2226.6844889322915
10000000.0 2226.6844889322915


In [31]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 2239.01025390625
100 2239.0091959635415
1000 2239.0104166666665
10000 2239.0723470052085
100000.0 2239.7661946614585
1000000.0 2299.219970703125
10000000.0 2659.0167643229165


In [32]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

10 12.325764973958485
100 12.32470703125
1000 12.325927734375
10000 12.38785807291697
100000.0 13.08170572916697
1000000.0 72.53548177083348
10000000.0 432.332275390625


In [ ]:
#  Best: 100  12.32470703125
# Close: 10   12.325764973958485
#        1000 12.325927734375

In [33]:
prev_results = dict()
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2
DECORRELATION_TAUS = [10, 100, 1000, 10000, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10]

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    prev_results[key] = []
    results[key] = []

    print(key)

    for seed in range(NUM_TRAINS):
        print(seed)
        
        model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            model_params={
                'decorrelation_tau': 0.01,  # best values
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )

        for reg in model.regularizers.data:
            print(f"{reg}: {model.regularizers[reg].tau}")

        result = fit_and_compute_scores(model, dataset)
        prev_results[key].append(result)

        
        good_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
        ]
        bad_topic_indices = [
            t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
        ]
        not_good_topic_indices = [
            t for t in TOPIC_INDICES if t not in good_topic_indices
        ]
        
        phi = model.get_phi()
        good_topic_names = [phi.columns[t] for t in good_topic_indices]
        bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
        not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]


        assert len(good_topic_names) > 0
        assert len(bad_topic_names) > 0

        print(len(good_topic_names), len(bad_topic_names), len(not_good_topic_names))


        fix_regularizer = FastFixPhiRegularizer(
            name='fix',
            parent_model=model._model,
            topic_names=good_topic_names,
        )
        
        bad_phi = model._model.get_phi()[bad_topic_names]
        bad_phi = deepcopy(bad_phi)
        decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_bad', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=bad_phi
        )
        
        good_phi = model._model.get_phi()[good_topic_names]
        good_phi = deepcopy(good_phi)
        decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
            name='ext_decorr_good', tau=decorrelation_tau,
            topic_names=not_good_topic_names,
            other_phi=good_phi
        )

        
        assert not hasattr(fix_regularizer, '_model')


        new_model = init_model_from_family(
            family=KnownModel.ARTM,
            dataset=dataset,
            main_modality=MAIN_MODALITY,
            num_topics=NUM_TOPICS,
            seed=seed,
            specific_topic_names=not_good_topic_names,
            model_params={
                'decorrelation_tau': 0.01,
                'smooth_bcg_tau': 0.05,
                'sparse_sp_tau': -0.05,
            }
        )
        custom_regularizers = {
            fix_regularizer.name: fix_regularizer,
            decorr_bad_regularizer.name: decorr_bad_regularizer,
            decorr_good_regularizer.name: decorr_good_regularizer,
        }

        new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
        
        for reg in new_model.regularizers.data:
            print(f"{reg}: {new_model.regularizers[reg].tau}")

        for reg_name, reg in custom_regularizers.items():
            print(f"{reg_name}: {reg.tau}")

        
        results[key].append(new_result)

10
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
6 2 14


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d08a8460>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96c5f1b910>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96e83a8820>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10
ext_decorr_good: 10
1
No spec topics
No spec topics


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d01bcfa0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96d0d60ee0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96d0d60490>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.06388419415493585
sparse_theta_sp: -0.29816015719918026
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000000.0
ext_decorr_good: 10000000000.0
2
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
7 3 13


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f96d05ce0d0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96d0ea5280>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f96d0ea5220>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.07371253171723367
sparse_theta_sp: -0.3440309506144388
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000000.0
ext_decorr_good: 10000000000.0


In [37]:
for k, r in prev_results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 2226.6844889322915
100 2226.6844889322915
1000 2226.6844889322915
10000 2226.6844889322915
100000.0 2226.6844889322915
1000000.0 2226.6844889322915
10000000.0 2226.6844889322915
100000000.0 2226.6844889322915
1000000000.0 2226.6844889322915
10000000000.0 2226.6844889322915


In [38]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)
    print(k, mean_ppl)

10 2239.0099283854165
100 2239.0099283854165
1000 2239.0100911458335
10000 2239.01025390625
100000.0 2239.0098470052085
1000000.0 2239.0165201822915
10000000.0 2239.1668294270835
100000000.0 2240.8199055989585
1000000000.0 2285.2738444010415
10000000000.0 2437.2290852864585


In [39]:
for k, r in results.items():
    mean_ppl = sum(v['scores']['perplexity'] for v in r) / len(r)

    prev_r = prev_results[k]
    prev_mean_ppl = sum(v['scores']['perplexity'] for v in prev_r) / len(prev_r)

    print(k, mean_ppl - prev_mean_ppl)

10 12.325439453125
100 12.325439453125
1000 12.32560221354197
10000 12.325764973958485
100000.0 12.32535807291697
1000000.0 12.33203125
10000000.0 12.48234049479197
100000000.0 14.13541666666697
1000000000.0 58.58935546875
10000000000.0 210.54459635416697


In [40]:
#  Best: 100000  12.32535807291697
# Close: 1000000 12.33203125
#        10000   12.325764973958485

# Edgy:
# 10000000.0 12.48234049479197
# 100000000.0 14.13541666666697
# 1000000000.0 58.58935546875

In [41]:
MAX_NUM_TRAINS

20

In [42]:
results

{10: [{'scores': {'perplexity': 2253.85498046875,
    'coherence_20': array([1.44551848]),
    'diversity_euclidean': 0.07102475422861072,
    'diversity_jensenshannon': 0.6992353616454047,
    'diversity_hellinger': 0.8179134374781466,
    'diversity_cosine': 0.8265317080905235},
   'topic_coherences': {0: 1.054240285921331,
    1: 1.453658536717904,
    2: 1.4334904461780176,
    3: 1.0063072833414675,
    4: 1.0096204089012255,
    5: 1.5539136077121654,
    6: 1.3994107963122735,
    7: 0.9778608908985307,
    8: 0.6195399396999054,
    9: 0.8006673680341132,
    10: 1.840761298075488,
    11: 1.2604398789491713,
    12: 1.7073209789137267,
    13: 1.8580910947101374,
    14: 1.4267776480811674,
    15: 1.6073534123690463,
    16: 2.0158025400053132,
    17: 2.636592011876535,
    18: 1.7554318141499257,
    19: 1.4930893269495578}},
  {'scores': {'perplexity': 2230.625244140625,
    'coherence_20': array([1.46466194]),
    'diversity_euclidean': 0.0683676049899362,
    'diversity_

In [43]:
new_result

{'scores': {'perplexity': 2461.866943359375,
  'coherence_20': array([1.85850209]),
  'diversity_euclidean': 0.07356362067834214,
  'diversity_jensenshannon': 0.7711511659469856,
  'diversity_hellinger': 0.9155164493873434,
  'diversity_cosine': 0.9444821285586293},
 'topic_coherences': {0: 1.7300523146588405,
  1: 1.5709401517076538,
  2: 1.9563085816159047,
  3: 1.6460933884449667,
  4: 2.26887264809252,
  5: 2.133835008205159,
  6: 2.107893087858345,
  7: 1.6728566636808413,
  8: 2.237799559639728,
  9: 1.8893206007995285,
  10: 1.9824795351569895,
  11: 2.159035393982212,
  12: 0.9739755322110191,
  13: 2.2229610002548172,
  14: 1.2609566862135837,
  15: 1.6770159675711767,
  16: 2.8450990542764627,
  17: 1.6517525523607473,
  18: 2.116133177601322,
  19: 1.066660946339064}}

In [44]:
fix_regularizer._topic_names

['topic_0',
 'topic_2',
 'topic_3',
 'topic_8',
 'topic_10',
 'topic_11',
 'topic_16']

In [45]:
del model

In [76]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

#  Best: 100  12.32470703125
# Close: 10   12.325764973958485
#        1000 12.325927734375

# Edgy:
# 100000.0 13.08170572916697
# 1000000.0 72.53548177083348

# DECORRELATION_TAUS = [100, 1000]  # Best
DECORRELATION_TAUS = [100000, 1000000]  # Dangerous

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

100000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
num_topics: {'good': 6, 'bad': 2, 'not_good': 14, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f12388778e0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123a6ecf40>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123b1d57c0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 100000
ext_decorr_good: 100000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 8, 'bad': 3, 'not_good': 12, 'total_bad': 5}
2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f12399e5160>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123a6ecf40>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123ab21670>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
num_topics: {'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 15}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f121d4a5f40>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123aa4ed30>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123af1cdc0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
SUCCESS: less bad topics
SUCCESS: no bad topics!
num_topics: {'good': 19, 'bad': 0, 'not_good': 1, 'total_bad': 15}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f123a6ecf40>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f1218b9d5b0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f121d390430>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
DOWNFALL: more bad topics...
num_topics: {'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 16}


In [77]:
1

1

In [78]:
results.keys()

dict_keys([100000, 1000000])

In [79]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [29]:
import json

SAVE_FOLDER = 'results/20newsgroups'

! mkdir -p $SAVE_FOLDER

In [30]:
results[100][0]

{'scores': {'perplexity': 2236.79248046875,
  'coherence_20': 1.4469243975267287,
  'diversity_euclidean': 0.06941780360242247,
  'diversity_jensenshannon': 0.6988145697570417,
  'diversity_hellinger': 0.8168150929911163,
  'diversity_cosine': 0.8307823760072601},
 'topic_coherences': {0: 1.1491291917105928,
  1: 1.5517711527505458,
  2: 1.405924126520918,
  3: 1.0506564618250258,
  4: 1.1270220165324627,
  5: 1.5564847213025248,
  6: 1.3764434481297354,
  7: 0.9760657812912945,
  8: 0.6237645959171805,
  9: 0.7385423660674926,
  10: 1.840761298075488,
  11: 1.330514926154629,
  12: 1.7073209789137267,
  13: 1.8580910947101374,
  14: 1.3211340090044137,
  15: 1.4857567187040455,
  16: 2.0158025400053132,
  17: 2.636592011876535,
  18: 1.7554318141499257,
  19: 1.4312786968925868},
 'num_topics': {'good': 6, 'bad': 2, 'not_good': 14, 'total_bad': 2}}

In [80]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [44]:
! ls $SAVE_FOLDER

decorrelation.json   iterative2_1000000.json  plsa.json
iterative_1000.json  iterative2_100000.json   sparse.json
iterative_100.json   lda.json		      tless.json


In [46]:
! tail -n 50 $SAVE_FOLDER/iterative_100.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.683349609375,
            "coherence_20": 1.5343349867118177,
            "diversity_euclidean": 0.07490313695236982,
            "diversity_jensenshannon": 0.709611598639548,
            "diversity_hellinger": 0.8321301188245224,
            "diversity_cosine": 0.8474179226005811
        },
        "topic_coherences": {
            "0": 0.9217079965108185,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.8541021951209471,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.9076733045710277,
            "8": 0.9002829919095333,
            "9": 1.6300

In [47]:
! tail -n 50 $SAVE_FOLDER/iterative_1000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.755859375,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07488431260088221,
            "diversity_jensenshannon": 0.7099821904985755,
            "diversity_hellinger": 0.8326169812227255,
            "diversity_cosine": 0.8487093680907188
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801523,
            "6": 1.8054658007396136,
            "7": 0.9076733045710277,
            "8": 0.9312208318641825,
            "9": 1.630026790

In [ ]:
results.keys()

In [ ]:
results[1000][-1]

In [81]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

#  Best: 100000  12.32535807291697
# Close: 1000000 12.33203125
#        10000   12.325764973958485

# Edgy:
# 10000000.0 12.48234049479197
# 100000000.0 14.13541666666697
# 1000000000.0 58.58935546875

# DECORRELATION_TAUS = [100000, 1000000]  # Best
DECORRELATION_TAUS = [10000000, 100000000, 1000000000]  # Dangerous

for decorrelation_tau in DECORRELATION_TAUS:
    key = decorrelation_tau
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:

            fix_regularizer = FastFixPhiRegularizer(
                name='fix',
                parent_model=prev_model._model,
                topic_names=good_topic_names,
            )
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)
            decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_bad', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=bad_phi
            )
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)
            decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                name='ext_decorr_good', tau=decorrelation_tau,
                topic_names=not_good_topic_names,
                other_phi=good_phi
            )
        
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
            custom_regularizers = {
                fix_regularizer.name: fix_regularizer,
                decorr_bad_regularizer.name: decorr_bad_regularizer,
                decorr_good_regularizer.name: decorr_good_regularizer,
            }
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(new_good_topic_names) > 0
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            assert set(good_topic_names) <= set(new_good_topic_names)
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

10000000
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
num_topics: {'good': 6, 'bad': 2, 'not_good': 14, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f123b2d60a0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f123ab218e0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f12188032b0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 10000000
ext_decorr_good: 10000000
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 8, 'bad': 3, 'not_good': 12, 'total_bad': 5}
2


...

{'fix': <__main__.FastFixPhiRegularizer object at 0x7f1238f0a0a0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f12383703a0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f1238ad7340>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000000
ext_decorr_good: 1000000000
num_topics: {'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 13}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f1239d4b7f0>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f1239bf0880>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f121880d7c0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000000
ext_decorr_good: 1000000000
num_topics: {'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 14}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'fix': <__main__.FastFixPhiRegularizer object at 0x7f123aab4d90>, 'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f12188445b0>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f123aece940>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.9582629123240377
sparse_theta_sp: -4.472402357987704
decorrelation: 0.01
fix: 1000000000
ext_decorr_bad: 1000000000
ext_decorr_good: 1000000000
num_topics: {'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 15}


In [87]:
1

1

In [90]:
results.keys()

dict_keys([10000000, 100000000, 1000000000])

In [91]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [ ]:
1

In [92]:
for k, r in results.items():
    with open(SAVE_FOLDER + f'/iterative2_{int(k)}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [93]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [ ]:
view_model(prev_model, dataset)

In [48]:
! tail -n 50 $SAVE_FOLDER/iterative2_100000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.692138671875,
            "coherence_20": 1.5343349867118177,
            "diversity_euclidean": 0.07488779816367575,
            "diversity_jensenshannon": 0.7096028483722486,
            "diversity_hellinger": 0.832122429571492,
            "diversity_cosine": 0.8473827547639936
        },
        "topic_coherences": {
            "0": 0.9217079965108185,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.8541021951209471,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.9076733045710277,
            "8": 0.9002829919095333,
            "9": 1.6300

In [49]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.75244140625,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07473976332049242,
            "diversity_jensenshannon": 0.7098907759986864,
            "diversity_hellinger": 0.8325292643052555,
            "diversity_cosine": 0.8483379385827017
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.907673304571028,
            "8": 0.9312208318641825,
            "9": 1.63002679

In [94]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [98]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.75244140625,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07473976332049242,
            "diversity_jensenshannon": 0.7098907759986864,
            "diversity_hellinger": 0.8325292643052555,
            "diversity_cosine": 0.8483379385827017
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.907673304571028,
            "8": 0.9312208318641825,
            "9": 1.63002679

## Ablation Study

In [102]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [104]:
! tail -n 50 $SAVE_FOLDER/iterative_1000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.755859375,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07488431260088221,
            "diversity_jensenshannon": 0.7099821904985755,
            "diversity_hellinger": 0.8326169812227255,
            "diversity_cosine": 0.8487093680907188
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801523,
            "6": 1.8054658007396136,
            "7": 0.9076733045710277,
            "8": 0.9312208318641825,
            "9": 1.630026790

In [105]:
! tail -n 50 $SAVE_FOLDER/iterative_100000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 19,
            "bad": 1,
            "not_good": 1,
            "total_bad": 23
        }
    },
    {
        "scores": {
            "perplexity": 2524.498046875,
            "coherence_20": 1.8160830373355887,
            "diversity_euclidean": 0.09386924139489292,
            "diversity_jensenshannon": 0.7519331493667284,
            "diversity_hellinger": 0.8879142519887491,
            "diversity_cosine": 0.91592813474032
        },
        "topic_coherences": {
            "0": 1.7445017980379762,
            "1": 2.057984232106506,
            "2": 1.7450314186955223,
            "3": 0.6671540546965984,
            "4": 2.29197749690007,
            "5": 1.6710892938824142,
            "6": 1.8767654446024988,
            "7": 1.76869936619555,
            "8": 2.215910718477992,
            "9": 1.63002679049288

In [106]:
! tail -n 50 $SAVE_FOLDER/iterative_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.6457005218273515
        },
        "num_topics": {
            "good": 19,
            "bad": 0,
            "not_good": 1,
            "total_bad": 15
        }
    },
    {
        "scores": {
            "perplexity": 2509.45068359375,
            "coherence_20": 1.8343696152756745,
            "diversity_euclidean": 0.10217171248725132,
            "diversity_jensenshannon": 0.7695284135036325,
            "diversity_hellinger": 0.9123624284297017,
            "diversity_cosine": 0.9442764712790213
        },
        "topic_coherences": {
            "0": 0.6043036512713817,
            "1": 1.85624161842734,
            "2": 1.8789415533863691,
            "3": 2.0087784467994148,
            "4": 1.837319679382072,
            "5": 1.8946013416131207,
            "6": 1.90155438803718,
            "7": 1.6737924853770243,
            "8": 1.8444252589807613,
            "9": 2.11380410

In [ ]:
DECORRELATION_TAUS = [100000, 1000000]  # Dangerous
DECORRELATION_TAUS = [10000000, 100000000, 1000000000]  # Dangerous

In [100]:
! ls $SAVE_FOLDER/ablation_study

iterative_1000_0-0-1.json  iterative2_1000000_0-0-1.json
iterative_1000_0-1-0.json  iterative2_1000000_0-1-0.json
iterative_1000_0-1-1.json  iterative2_1000000_0-1-1.json
iterative_1000_1-0-0.json  iterative2_1000000_1-0-0.json
iterative_1000_1-0-1.json  iterative2_1000000_1-0-1.json
iterative_1000_1-1-0.json  iterative2_1000000_1-1-0.json


In [113]:
# DECORRELATION_TAU = 1000
DECORRELATION_TAU = 1000000  # new best # TODO: maybe even higher?

ALL_PARAMS = [
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
]

In [114]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer

for params in ALL_PARAMS:
    key = params
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

(0, 1, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
num_topics: {'good': 6, 'bad': 2, 'not_good': 14, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f123aaba400>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f1239ebe2e0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
ext_decorr_bad: 1000000
ext_decorr_good: 1000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 12, 'bad': 3, 'not_good': 8, 'total_bad': 5}
2


...

{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f1238fc4d00>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.07985524269366981
sparse_theta_sp: -0.37270019649897534
decorrelation: 0.01
ext_decorr_good: 1000000
DOWNFALL: some good topics lost...
DOWNFALL: more bad topics...
num_topics: {'good': 2, 'bad': 4, 'not_good': 18, 'total_bad': 55}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f121d2d8df0>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.05323682846244655
sparse_theta_sp: -0.24846679766598356
decorrelation: 0.01
ext_decorr_good: 1000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
num_topics: {'good': 4, 'bad': 4, 'not_good': 16, 'total_bad': 59}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer object at 0x7f1238916160>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.05989143202025236
sparse_theta_sp: -0.2795251473742315
decorrelation: 0.01
ext_decorr_good: 1000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
num_topics: {'good': 8, 'bad': 4, 'not_good': 12, 'total_bad': 63}


In [115]:
1

1

In [116]:
! ls results/20newsgroups

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [117]:
! mkdir -p results/20newsgroups/ablation_study

In [118]:
results.keys()

dict_keys([(0, 1, 1), (1, 0, 1), (1, 1, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)])

In [119]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [120]:
'-'.join(str(i) for i in k)

'0-0-1'

In [121]:
for k, r in results.items():
    output_k = '-'.join(str(i) for i in k)
    with open(SAVE_FOLDER + f'/ablation_study/iterative_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [122]:
! ls results/20newsgroups/ablation_study

iterative_1000000_0-0-1.json  iterative_1000_1-0-0.json
iterative_1000000_0-1-0.json  iterative_1000_1-0-1.json
iterative_1000000_0-1-1.json  iterative_1000_1-1-0.json
iterative_1000000_1-0-0.json  iterative2_1000000_0-0-1.json
iterative_1000000_1-0-1.json  iterative2_1000000_0-1-0.json
iterative_1000000_1-1-0.json  iterative2_1000000_0-1-1.json
iterative_1000_0-0-1.json     iterative2_1000000_1-0-0.json
iterative_1000_0-1-0.json     iterative2_1000000_1-0-1.json
iterative_1000_0-1-1.json     iterative2_1000000_1-1-0.json


In [123]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()

(0, 1, 1)
{'perplexity': 2298.86474609375, 'coherence_20': 1.7097599620715205, 'diversity_euclidean': 0.07696227654850847, 'diversity_jensenshannon': 0.75997374559108, 'diversity_hellinger': 0.8996516085590787, 'diversity_cosine': 0.8922071526632959}
{'good': 8, 'bad': 3, 'not_good': 12, 'total_bad': 68}

(1, 0, 1)
{'perplexity': 2464.59130859375, 'coherence_20': 1.8222803403386731, 'diversity_euclidean': 0.09665751707718807, 'diversity_jensenshannon': 0.7558104290820944, 'diversity_hellinger': 0.8931846991572905, 'diversity_cosine': 0.9294582819017179}
{'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 21}

(1, 1, 0)
{'perplexity': 2476.902099609375, 'coherence_20': 1.7674292440536103, 'diversity_euclidean': 0.09821210131545323, 'diversity_jensenshannon': 0.7519702519427854, 'diversity_hellinger': 0.8875734457352042, 'diversity_cosine': 0.9115068086898529}
{'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 12}

(1, 0, 0)
{'perplexity': 2399.752197265625, 'coherence_20': 1.532591468855

In [124]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [107]:
! ls $SAVE_FOLDER

ablation_study		iterative_100.json	    iterative2_100000.json
decorrelation.json	iterative2_1000000000.json  lda.json
iterative_1000000.json	iterative2_100000000.json   plsa.json
iterative_100000.json	iterative2_10000000.json    sparse.json
iterative_1000.json	iterative2_1000000.json     tless.json


In [108]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 4,
            "not_good": 7,
            "total_bad": 54
        }
    },
    {
        "scores": {
            "perplexity": 2399.75244140625,
            "coherence_20": 1.5422349498469412,
            "diversity_euclidean": 0.07473976332049242,
            "diversity_jensenshannon": 0.7098907759986864,
            "diversity_hellinger": 0.8325292643052555,
            "diversity_cosine": 0.8483379385827017
        },
        "topic_coherences": {
            "0": 1.00113211617019,
            "1": 1.3621768174895768,
            "2": 1.6694630947055196,
            "3": 0.901739498209401,
            "4": 0.9697594159537758,
            "5": 0.6067957005801524,
            "6": 1.8054658007396136,
            "7": 0.907673304571028,
            "8": 0.9312208318641825,
            "9": 1.63002679

In [109]:
! tail -n 50 $SAVE_FOLDER/iterative2_10000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 13,
            "bad": 2,
            "not_good": 7,
            "total_bad": 45
        }
    },
    {
        "scores": {
            "perplexity": 2403.99267578125,
            "coherence_20": 1.573263970374759,
            "diversity_euclidean": 0.07375830564020532,
            "diversity_jensenshannon": 0.7124094332529214,
            "diversity_hellinger": 0.8359994991184351,
            "diversity_cosine": 0.855706460551422
        },
        "topic_coherences": {
            "0": 1.1082963221453308,
            "1": 1.4004677404363155,
            "2": 1.6549948867753843,
            "3": 0.9853736724876744,
            "4": 1.1166110279043346,
            "5": 0.6067957005801523,
            "6": 1.8054658007396136,
            "7": 1.0752934564635988,
            "8": 0.9390066126540663,
            "9": 1.630026

In [110]:
! tail -n 50 $SAVE_FOLDER/iterative2_100000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.640629444035244
        },
        "num_topics": {
            "good": 19,
            "bad": 1,
            "not_good": 1,
            "total_bad": 17
        }
    },
    {
        "scores": {
            "perplexity": 2536.84521484375,
            "coherence_20": 1.7657066593419635,
            "diversity_euclidean": 0.08474110227762996,
            "diversity_jensenshannon": 0.7364641673959812,
            "diversity_hellinger": 0.8673032015098073,
            "diversity_cosine": 0.8908660854388641
        },
        "topic_coherences": {
            "0": 1.6603571660847984,
            "1": 1.8608120102679044,
            "2": 0.6321317616434708,
            "3": 2.08647789775495,
            "4": 1.8032283276088727,
            "5": 1.696860620500633,
            "6": 1.8054658007396138,
            "7": 1.651973818044545,
            "8": 1.7787379471054157,
            "9": 1.63002679

In [112]:
! tail -n 50 $SAVE_FOLDER/iterative2_1000000000.json

            "17": 2.636592011876535,
            "18": 1.7554318141499257,
            "19": 1.6167513780664968
        },
        "num_topics": {
            "good": 19,
            "bad": 1,
            "not_good": 1,
            "total_bad": 14
        }
    },
    {
        "scores": {
            "perplexity": 2509.298095703125,
            "coherence_20": 1.8004192080579677,
            "diversity_euclidean": 0.09672882730245023,
            "diversity_jensenshannon": 0.7499680063404501,
            "diversity_hellinger": 0.8856458799229798,
            "diversity_cosine": 0.9193192437631968
        },
        "topic_coherences": {
            "0": 1.6834217755146534,
            "1": 1.6636143961405991,
            "2": 1.9126025217315148,
            "3": 1.7636570949116437,
            "4": 1.6862600158085381,
            "5": 1.9193798353133522,
            "6": 2.2642665670036526,
            "7": 0.6104632760028169,
            "8": 2.091593575026202,
            "9": 1.824

In [ ]:
DECORRELATION_TAUS = [100000, 1000000]  # Dangerous
DECORRELATION_TAUS = [10000000, 100000000, 1000000000]  # Dangerous

In [125]:
# DECORRELATION_TAU = 1000000
DECORRELATION_TAU = 1000000000  # new best # TODO: maybe even higher?

ALL_PARAMS = [
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0),

    (1, 0, 0),
    (0, 1, 0),
    (0, 0, 1),
]

In [126]:
results = dict()

DECORRELATOR_REGULARIZER_CLASS = DecorrelatorWithOtherPhiRegularizer2

#  Best: 100000000.0 55.63102213541697
# Close: 10000000.0 55.890380859375

for params in ALL_PARAMS:
    key = params
    results[key] = []

    print(key)

    prev_model = None
    good_topic_names = list()
    bad_topic_names = None
    not_good_topic_names = None
    bad_phi = None
    seed = 0

    while seed < MAX_NUM_TRAINS and len(good_topic_names) < NUM_TOPICS:
        print(seed)

        if seed == 0:
            prev_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                model_params={
                    'decorrelation_tau': 0.01,  # best values
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )

            for reg in prev_model.regularizers.data:
                print(f"{reg}: {prev_model.regularizers[reg].tau}")
    
            result = fit_and_compute_scores(prev_model, dataset)
            results[key].append(result)

            assert len(results[key]) == seed + 1
    
            
            good_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = prev_model.get_phi()
            good_topic_names = [phi.columns[t] for t in good_topic_indices]
            bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            assert len(good_topic_names) > 0
            assert len(bad_topic_names) > 0
            assert set(bad_topic_names) <= set(not_good_topic_names)
            assert not any(t in not_good_topic_names for t in good_topic_names)
            assert len(good_topic_names) + len(not_good_topic_names) == NUM_TOPICS

            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': len(bad_topic_names),
            }

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1
            
            del result, phi
            model = None

        else:
            custom_regularizers = dict()
            
            if params[0]:
                fix_regularizer = FastFixPhiRegularizer(
                    name='fix',
                    parent_model=prev_model._model,
                    topic_names=good_topic_names,
                )
                custom_regularizers[fix_regularizer.name] = fix_regularizer
            else:
                fix_regularizer = None
            
            cur_bad_phi = prev_model._model.get_phi()[bad_topic_names]

            if bad_phi is None:
                bad_phi = cur_bad_phi
            else:
                # bad_phi.rename(
                #     columns={n: f'm1_{n}' for n in bad_topic_names}, inplace=True
                # )
                bad_phi = pd.concat([bad_phi, cur_bad_phi], axis=1)
    
            bad_phi = deepcopy(bad_phi)

            if params[1]:
                decorr_bad_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_bad', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=bad_phi
                )
                custom_regularizers[decorr_bad_regularizer.name] = decorr_bad_regularizer
            else:
                decorr_bad_regularizer = None
            
            good_phi = prev_model._model.get_phi()[good_topic_names]
            good_phi = deepcopy(good_phi)

            if params[2]:
                decorr_good_regularizer = DECORRELATOR_REGULARIZER_CLASS(
                    name='ext_decorr_good', tau=DECORRELATION_TAU,
                    topic_names=not_good_topic_names,
                    other_phi=good_phi
                )
                custom_regularizers[decorr_good_regularizer.name] = decorr_good_regularizer
            else:
                decorr_good_regularizer = None
    
            new_model = init_model_from_family(
                family=KnownModel.ARTM,
                dataset=dataset,
                main_modality=MAIN_MODALITY,
                num_topics=NUM_TOPICS,
                seed=seed,
                specific_topic_names=not_good_topic_names,
                model_params={
                    'decorrelation_tau': 0.01,
                    'smooth_bcg_tau': 0.05,
                    'sparse_sp_tau': -0.05,
                }
            )
    
            new_result = fit_and_compute_scores(new_model, dataset, custom_regularizers=custom_regularizers)
            
            for reg in new_model.regularizers.data:
                print(f"{reg}: {new_model.regularizers[reg].tau}")
    
            for reg_name, reg in custom_regularizers.items():
                print(f"{reg_name}: {reg.tau}")
    
            
            results[key].append(new_result)

            assert len(results[key]) == seed + 1


            
            good_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_good(c)
            ]
            bad_topic_indices = [
                t for t, c in new_result['topic_coherences'].items() if t in TOPIC_INDICES and is_bad(c)
            ]
            not_good_topic_indices = [
                t for t in TOPIC_INDICES if t not in good_topic_indices
            ]
            
            phi = new_model.get_phi()
            new_good_topic_names = [phi.columns[t] for t in good_topic_indices]
            new_bad_topic_names = [phi.columns[t] for t in bad_topic_indices]
            new_not_good_topic_names = [phi.columns[t] for t in not_good_topic_indices]

            # assert len(new_good_topic_names) > 0
            if len(new_good_topic_names) == 0:
                print('DOWNFALL: no good topics...')
            # assert len(new_bad_topic_names) > 0
            assert set(new_bad_topic_names) <= set(new_not_good_topic_names)
            assert not any(t in new_not_good_topic_names for t in new_good_topic_names)
            assert len(new_good_topic_names) + len(new_not_good_topic_names) == NUM_TOPICS

            # assert set(good_topic_names) <= set(new_good_topic_names)
            if not (set(good_topic_names) <= set(new_good_topic_names)):
                print('DOWNFALL: some good topics lost...')
            # assert len(new_bad_topic_names) <= len(bad_topic_names)

            if len(new_good_topic_names) > len(good_topic_names):
                print('SUCCESS: more good topics')
            if len(new_bad_topic_names) < len(bad_topic_names):
                print('SUCCESS: less bad topics')
            if len(new_bad_topic_names) > len(bad_topic_names):
                print('DOWNFALL: more bad topics...')
            if len(new_bad_topic_names) == 0:
                print('SUCCESS: no bad topics!')
    
            good_topic_names = new_good_topic_names
            bad_topic_names = new_bad_topic_names
            not_good_topic_names = new_not_good_topic_names

            
            assert 'num_topics' not in results[key][-1]

            results[key][-1]['num_topics'] = {
                'good': len(good_topic_names),
                'bad': len(bad_topic_names),
                'not_good': len(not_good_topic_names),
                'total_bad': bad_phi.shape[1] + len(bad_topic_names),
            }

            prev_model = new_model

            print(f"num_topics: {results[key][-1]['num_topics']}")

            seed += 1

(0, 1, 1)
0
No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.047913145616201885
sparse_theta_sp: -0.2236201178993852
decorrelation: 0.01
None
num_topics: {'good': 6, 'bad': 2, 'not_good': 14, 'total_bad': 2}
1


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_bad': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f123ab57880>, 'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f1239de6850>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
ext_decorr_bad: 1000000000
ext_decorr_good: 1000000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 7, 'bad': 4, 'not_good': 13, 'total_bad': 6}
2


...

{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f123ad6a730>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.056368406607296334
sparse_theta_sp: -0.2630824916463355
decorrelation: 0.01
ext_decorr_good: 1000000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
SUCCESS: less bad topics
num_topics: {'good': 4, 'bad': 3, 'not_good': 16, 'total_bad': 74}
18


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f121d364e50>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.05989143202025236
sparse_theta_sp: -0.2795251473742315
decorrelation: 0.01
ext_decorr_good: 1000000000
DOWNFALL: some good topics lost...
SUCCESS: more good topics
DOWNFALL: more bad topics...
num_topics: {'good': 6, 'bad': 5, 'not_good': 14, 'total_bad': 79}
19


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning:

Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!



{'ext_decorr_good': <__main__.DecorrelatorWithOtherPhiRegularizer2 object at 0x7f123ad6a730>}
smooth_phi_bcg: 1.0591326925686735
smooth_theta_bcg: 4.943181553565358
sparse_phi_sp: -0.0684473508802884
sparse_theta_sp: -0.319457311284836
decorrelation: 0.01
ext_decorr_good: 1000000000
DOWNFALL: some good topics lost...
SUCCESS: less bad topics
num_topics: {'good': 4, 'bad': 4, 'not_good': 16, 'total_bad': 83}


In [136]:
1

1

In [128]:
! ls results/20newsgroups/ablation_study/

iterative_1000000_0-0-1.json  iterative_1000_1-0-0.json
iterative_1000000_0-1-0.json  iterative_1000_1-0-1.json
iterative_1000000_0-1-1.json  iterative_1000_1-1-0.json
iterative_1000000_1-0-0.json  iterative2_1000000_0-0-1.json
iterative_1000000_1-0-1.json  iterative2_1000000_0-1-0.json
iterative_1000000_1-1-0.json  iterative2_1000000_0-1-1.json
iterative_1000_0-0-1.json     iterative2_1000000_1-0-0.json
iterative_1000_0-1-0.json     iterative2_1000000_1-0-1.json
iterative_1000_0-1-1.json     iterative2_1000000_1-1-0.json


In [129]:
DECORRELATION_TAU

1000000000

In [130]:
results.keys()

dict_keys([(0, 1, 1), (1, 0, 1), (1, 1, 0), (1, 0, 0), (0, 1, 0), (0, 0, 1)])

In [131]:
for k, r in results.items():
    for s in r:
        s['scores']['coherence_20'] = float(s['scores']['coherence_20'])

In [132]:
for k, r in results.items():
    output_k = '-'.join(str(i) for i in k)
    with open(SAVE_FOLDER + f'/ablation_study/iterative2_{DECORRELATION_TAU}_{output_k}.json', 'w') as f:
        f.write(
            json.dumps(r, indent=4)
        )

In [133]:
! ls results/20newsgroups/ablation_study

iterative_1000000_0-0-1.json  iterative2_1000000000_0-0-1.json
iterative_1000000_0-1-0.json  iterative2_1000000000_0-1-0.json
iterative_1000000_0-1-1.json  iterative2_1000000000_0-1-1.json
iterative_1000000_1-0-0.json  iterative2_1000000000_1-0-0.json
iterative_1000000_1-0-1.json  iterative2_1000000000_1-0-1.json
iterative_1000000_1-1-0.json  iterative2_1000000000_1-1-0.json
iterative_1000_0-0-1.json     iterative2_1000000_0-0-1.json
iterative_1000_0-1-0.json     iterative2_1000000_0-1-0.json
iterative_1000_0-1-1.json     iterative2_1000000_0-1-1.json
iterative_1000_1-0-0.json     iterative2_1000000_1-0-0.json
iterative_1000_1-0-1.json     iterative2_1000000_1-0-1.json
iterative_1000_1-1-0.json     iterative2_1000000_1-1-0.json


In [134]:
for k, r in results.items():
    print(k)
    print(r[-1]['scores'])
    print(r[-1]['num_topics'])
    print()

(0, 1, 1)
{'perplexity': 2378.796630859375, 'coherence_20': 1.7140309294414489, 'diversity_euclidean': 0.06756999250647623, 'diversity_jensenshannon': 0.7658887477473149, 'diversity_hellinger': 0.9073563482705373, 'diversity_cosine': 0.9247700426211884}
{'good': 12, 'bad': 4, 'not_good': 8, 'total_bad': 85}

(1, 0, 1)
{'perplexity': 2478.213623046875, 'coherence_20': 1.8082727927394142, 'diversity_euclidean': 0.08448469964980418, 'diversity_jensenshannon': 0.7407933136237574, 'diversity_hellinger': 0.8731731292801453, 'diversity_cosine': 0.9070294785718593}
{'good': 19, 'bad': 1, 'not_good': 1, 'total_bad': 24}

(1, 1, 0)
{'perplexity': 2534.749755859375, 'coherence_20': 1.8346049911182514, 'diversity_euclidean': 0.0876718756558992, 'diversity_jensenshannon': 0.7408455109076264, 'diversity_hellinger': 0.8738754706112992, 'diversity_cosine': 0.8998250257543015}
{'good': 18, 'bad': 0, 'not_good': 2, 'total_bad': 2}

(1, 0, 0)
{'perplexity': 2399.751953125, 'coherence_20': 1.5325914688550